# 98 — Dashboard export

Write `dashboard_data.parquet` for the React dashboard. Layout: one row per land grid cell, with

- underscore-prefixed metadata columns (`_lat`, `_lon`, `_country_code`, `_continent`, `_region_code`, `_region_name`) that the dashboard treats as non-scoring context,
- every percentile-normalized layer from `normalized.nc` **except** `temperature_pleasantness`, `precipitation_balance`, and `population_density` — those three are replaced by profile-based variants,
- for each of the three replaced layers, one column per profile in `TEMP_PROFILES` / `PRECIP_PROFILES` / `DENSITY_PROFILES`, suffixed `_p1`, `_p2`, …, and percentile-normalized so they share the same `[0, 1]` uniform distribution as the other layers.

In [40]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import rankdata

from common import (
    DENSITY_PROFILES,
    PRECIP_PROFILES,
    PROCESSED_DIR,
    TEMP_PROFILES,
    compute_population_density_score,
    compute_precipitation_balance,
    compute_temperature_pleasantness,
    load_raw_scoring_inputs,
)

## Profiles

Imported from `common.py` so this notebook and `99_atlas_map.ipynb` stay in sync. `(ideal, tolerance)` for temperature and precipitation; `((min, max), tolerance_decades)` for density where `None` means unbounded on that side — wilderness is a pure upper bound (empty cells still score 1), urban a pure lower bound.

## Helpers

`percentile_rank` mirrors the transform in `92_normalization.ipynb` so the profile columns share the same uniform `[0, 1]` distribution as the base layers already in `normalized.nc`.

In [41]:
def percentile_rank(da):
    """Rank-transform a DataArray to percentiles in ``[0, 1]``.

    Ties are averaged. NaN cells are preserved and excluded from the ranking.
    Shares dims, coords and name with the input so it can be dropped into any
    consumer that previously took the raw layer.
    """
    values = da.values.astype(float)
    flat = values.ravel()
    mask = np.isfinite(flat)
    out = np.full_like(flat, np.nan)
    n = int(mask.sum())
    if n > 1:
        out[mask] = (rankdata(flat[mask], method='average') - 1) / (n - 1)
    elif n == 1:
        out[mask] = 0.0
    return da.copy(data=out.reshape(values.shape))

## Base layers

Start from `normalized.nc` (percentile-transformed by `92_normalization.ipynb`) and drop the three layers we're replacing with profile-based variants.

In [42]:
REPLACED = ('temperature_pleasantness', 'precipitation_balance', 'population_density')

ds = xr.open_dataset(PROCESSED_DIR / 'normalized.nc')
base = ds.drop_vars([v for v in REPLACED if v in ds.data_vars])
list(base.data_vars)

['sea_proximity',
 'terrain_ruggedness',
 'sun_hours',
 'annual_greenness',
 'climate_vulnerability',
 'natural_disaster_risk',
 'air_quality',
 'urbanity',
 'internet_connectivity',
 'healthcare_access',
 'income',
 'cost_of_living',
 'crime_rate',
 'human_freedom',
 'corruption']

## Profile layers

For each profile, compute the raw comfort score with the helper in `common.py`, then percentile-rank so the result is comparable to the other columns. Column names are `<base>_p1`, `_p2`, … in dict-iteration order — the human-readable labels stay in this notebook (and are mirrored in the dashboard).

In [43]:
temp_da, precip_da, density_da = load_raw_scoring_inputs()

profile_layers = {}

for i, (label, (ideal, tol)) in enumerate(TEMP_PROFILES.items(), start=1):
    score = compute_temperature_pleasantness(temp_da, ideal_temp=ideal, tolerance=tol)
    profile_layers[f'temperature_pleasantness_p{i}'] = percentile_rank(score)
    print(f'temperature_pleasantness_p{i}  <- {label}')

for i, (label, (ideal, tol)) in enumerate(PRECIP_PROFILES.items(), start=1):
    score = compute_precipitation_balance(precip_da, ideal_monthly_mm=ideal, tolerance_mm=tol)
    profile_layers[f'precipitation_balance_p{i}'] = percentile_rank(score)
    print(f'precipitation_balance_p{i}     <- {label}')

for i, (label, (rng, tol_dec)) in enumerate(DENSITY_PROFILES.items(), start=1):
    score = compute_population_density_score(density_da, density_range=rng, tolerance_decades=tol_dec)
    profile_layers[f'population_density_p{i}'] = percentile_rank(score)
    print(f'population_density_p{i}        <- {label}')

temperature_pleasantness_p1  <- Icy (5°C ±8)
temperature_pleasantness_p2  <- Cool (15°C ±8)
temperature_pleasantness_p3  <- Temperate (20°C ±10)
temperature_pleasantness_p4  <- Warm (25°C ±8)
temperature_pleasantness_p5  <- Tropical (27°C ±5)
precipitation_balance_p1     <- Arid (20mm ±20)
precipitation_balance_p2     <- Balanced (80mm ±60)
precipitation_balance_p3     <- Wet (150mm ±60)
precipitation_balance_p4     <- Very wet (250mm ±80)
population_density_p1        <- Wilderness (≤1/km² ±1dec)
population_density_p2        <- Rural (10–300/km² ±1dec)
population_density_p3        <- Suburban (200–1500/km² ±1dec)
population_density_p4        <- Urban (≥1500/km² ±1dec)


## Merge and flatten

Combine the base and profile layers on the shared `(lat, lon)` grid, flatten to one row per cell, and drop cells that are NaN across every value column (pure ocean / coverage gaps) before the spatial join.

In [44]:
combined = xr.Dataset({**base.data_vars, **profile_layers})
df = combined.to_dataframe().reset_index()

value_cols = [c for c in df.columns if c not in ('lat', 'lon')]
df = df.dropna(subset=value_cols, how='all').reset_index(drop=True)
len(df)

68294

How many values are nan in every column after all operations?

In [47]:
data_columns = [c for c in df.columns if c not in ('lat', 'lon')]
nan_count_per_column = [df[c].isna().sum() for c in data_columns]
for name, count in zip(data_columns, nan_count_per_column):
    print(f"{name}: {count}")

sea_proximity: 0
terrain_ruggedness: 14
sun_hours: 34
annual_greenness: 2606
climate_vulnerability: 4158
natural_disaster_risk: 1512
air_quality: 8888
urbanity: 0
internet_connectivity: 37346
healthcare_access: 60
income: 368
cost_of_living: 831
crime_rate: 3945
human_freedom: 4023
corruption: 4023
temperature_pleasantness_p1: 20
temperature_pleasantness_p2: 20
temperature_pleasantness_p3: 20
temperature_pleasantness_p4: 20
temperature_pleasantness_p5: 20
precipitation_balance_p1: 20
precipitation_balance_p2: 20
precipitation_balance_p3: 20
precipitation_balance_p4: 20
population_density_p1: 0
population_density_p2: 0
population_density_p3: 0
population_density_p4: 0


## Region mask (admin-1)

Rasterise the [geoBoundaries CGAZ ADM1](https://www.geoboundaries.org/) states/provinces layer onto the atlas grid with `dominant_region_mask`, cached to `admin1_mask.nc` because the ~3600-region fractional-coverage pass is expensive. geoBoundaries (CC-BY 4.0) is used instead of Natural Earth 1:50m admin_1 because NE has good subdivisions only for the Americas and a handful of large countries — geoBoundaries provides global admin-1 coverage. Each cell gets a `region_code` synthesised as `{ISO3}-{shapeName}` (CGAZ has no ISO 3166-2 field) and a human-readable `region_name`. Cells outside every admin-1 polygon (ocean, disputed) stay as empty strings and become NaN after the join.

In [ ]:
from common import RAW_DIR, dominant_region_mask, download, load_grid

admin1_mask_path = PROCESSED_DIR / 'admin1_mask.nc'

if not admin1_mask_path.exists():
    import geopandas as gpd
    import regionmask

    variable_raw = RAW_DIR / 'region_mask'
    variable_raw.mkdir(parents=True, exist_ok=True)
    gpkg_path = variable_raw / 'geoBoundariesCGAZ_ADM1.gpkg'
    if not gpkg_path.exists():
        url = 'https://github.com/wmgeolab/geoBoundaries/raw/main/releaseData/CGAZ/geoBoundariesCGAZ_ADM1.gpkg'
        await download(url, gpkg_path)

    admin1 = gpd.read_file(gpkg_path).to_crs('EPSG:4326')
    admin1 = admin1[['shapeName', 'shapeGroup', 'geometry']].copy()
    # geoBoundaries CGAZ shipped ~24 shapeName values double-encoded (UTF-8
    # bytes read as Latin-1), e.g. "RegiÃ³n de Antofagasta". Re-encoding as
    # Latin-1 then decoding as UTF-8 recovers the original; names already
    # valid UTF-8 raise on the round-trip and are kept as-is.
    def _fix_mojibake(s):
        try:
            return s.encode('latin-1').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            return s
    admin1['shapeName'] = admin1['shapeName'].astype(str).map(_fix_mojibake)
    # CGAZ ADM1 has no ISO 3166-2 field, so synthesize a stable code from the
    # ISO3 country code + region name (e.g. "DEU-Bayern", "USA-California").
    admin1['code'] = (
        admin1['shapeGroup'].fillna('').astype(str) + '-' + admin1['shapeName'].fillna('').astype(str)
    )
    admin1 = admin1[admin1['shapeName'].notna()].reset_index(drop=True)
    print(f'{len(admin1)} admin-1 regions')

    grid = load_grid()
    regions = regionmask.from_geopandas(admin1, overlap=False)
    region_num = dominant_region_mask(regions, grid.lon, grid.lat)

    idx_arr = region_num.values
    valid = ~np.isnan(idx_arr)
    region_idx_i16 = np.where(valid, idx_arr, -1).astype('int16')

    mask_ds = xr.Dataset(
        {
            'region_idx': (('lat', 'lon'), region_idx_i16),
            'code': (('region_num',), admin1['code'].to_numpy()),
            'name': (('region_num',), admin1['shapeName'].to_numpy()),
        },
        coords={'lat': grid.lat, 'lon': grid.lon},
        attrs={
            'source': 'geoBoundaries CGAZ ADM1 via dominant_region_mask',
            'sentinel': '-1 in region_idx = no region (ocean / unlisted)',
        },
    )
    mask_ds.to_netcdf(admin1_mask_path)
    print(f'wrote {admin1_mask_path}')

admin1_ds = xr.open_dataset(admin1_mask_path)
_idx = admin1_ds['region_idx'].values
_codes = admin1_ds['code'].values
_names = admin1_ds['name'].values
_valid = _idx >= 0

region_code_arr = np.full(_idx.shape, '', dtype=object)
region_name_arr = np.full(_idx.shape, '', dtype=object)
region_code_arr[_valid] = _codes[_idx[_valid]]
region_name_arr[_valid] = _names[_idx[_valid]]

region_code_da = xr.DataArray(region_code_arr, coords={'lat': admin1_ds.lat, 'lon': admin1_ds.lon}, dims=('lat', 'lon'), name='region_code')
region_name_da = xr.DataArray(region_name_arr, coords={'lat': admin1_ds.lat, 'lon': admin1_ds.lon}, dims=('lat', 'lon'), name='region_name')

## Country, continent & region

Look up each cell's country from the cached `03_region_mask.ipynb` mask (dominant-coverage assignment, so island cells and coastal cells whose centre sits in the ocean still get a country). Continent comes from the accompanying `load_country_lookup()` table. Admin-1 region code + name are joined from the cell above. Ocean / disputed cells stay NaN in all of these columns.

In [ ]:
from common import load_country_lookup, load_country_mask

iso_per_cell = load_country_mask()
mask_df = (
    iso_per_cell.to_dataframe(name='country_code')
    .reset_index()
    .replace({'country_code': ''}, {'country_code': np.nan})
)

continent_map = dict(zip(load_country_lookup()['ISO3'], load_country_lookup()['CONTINENT']))
mask_df['continent'] = mask_df['country_code'].map(continent_map)

region_df = (
    xr.Dataset({'region_code': region_code_da, 'region_name': region_name_da})
    .to_dataframe()
    .reset_index()
    .replace({'region_code': '', 'region_name': ''}, {'region_code': np.nan, 'region_name': np.nan})
)
mask_df = mask_df.merge(region_df, on=['lat', 'lon'], how='left')

df = df.merge(mask_df[['lat', 'lon', 'country_code', 'continent', 'region_code', 'region_name']], on=['lat', 'lon'], how='left')

n_missing_country = df['country_code'].isna().sum()
n_missing_region = df['region_code'].isna().sum()
print(f'{len(df):,} cells; {n_missing_country:,} without a country, {n_missing_region:,} without a region')

## Save

Rename metadata columns to the underscore-prefixed form the dashboard expects (`_lat`, `_lon`, `_country_code`, `_continent`, `_region_code`, `_region_name`) and write.

In [ ]:
df = df.rename(columns={
    'lat': '_lat',
    'lon': '_lon',
    'country_code': '_country_code',
    'continent': '_continent',
    'region_code': '_region_code',
    'region_name': '_region_name',
})

out = PROCESSED_DIR / 'dashboard_data.parquet'
df.to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size / 1024**2:.1f} MB, {len(df):,} rows, {len(df.columns)} cols)')
df.head()

## Grid-cell polygons

Export one `Polygon` feature per row of `dashboard_data.parquet` as raw `[lon, lat]` corner coordinates (RFC 7946, WGS84). The dashboard reprojects them at render time — see `geometry.ts::transformCoordinates` / `buildCell`, which sample intermediate points along each edge before projecting so cells stay curved on the map.

Features are in parquet-row order; the dashboard joins each polygon to its scalar value by array index.

In [ ]:
# Half of the grid cell size in degrees (matches RES = 0.5 in the pipeline;
# cells are 0.5° squares centered on _lat / _lon).
HALF = 0.25


def build_cell(lon, lat):
    """Return a closed lon/lat ring (5 points, counter-clockwise) for a grid
    cell centered at (lon, lat).

    Coordinates are rounded to 2 decimal places — grid centers land on the
    0.5° grid so 2 dp is exact, and rounding also shields the emitted JSON
    from float-repr noise like ``45.24999999999``.
    """
    w = round(lon - HALF, 2)
    e = round(lon + HALF, 2)
    s = round(lat - HALF, 2)
    n = round(lat + HALF, 2)
    return [[w, s], [e, s], [e, n], [w, n], [w, s]]

In [ ]:
import json

# Feature order matches parquet row order — the dashboard joins by array index,
# so we don't duplicate the index into properties.
features = [
    {
        'type': 'Feature',
        'properties': {},
        'geometry': {
            'type': 'Polygon',
            'coordinates': [build_cell(lon, lat)],
        },
    }
    for lon, lat in zip(df['_lon'].to_numpy(), df['_lat'].to_numpy())
]

fc = {'type': 'FeatureCollection', 'features': features}

geo_out = PROCESSED_DIR / 'dashboard_cells.geojson'
geo_out.write_text(json.dumps(fc, separators=(',', ':')))
print(f'wrote {geo_out} ({geo_out.stat().st_size / 1024**2:.1f} MB, {len(features):,} features)')

wrote /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/processed/dashboard_cells.geojson (9.6 MB, 68,294 features)
